## Notebook de lectura de celdas sudoku

In [15]:
import matplotlib.pyplot as plt
import pandas as pd
import cv2
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split


In [29]:
#lo cargo como texto
df = pd.read_csv(
    "sudoku.csv",
    dtype={
        "puzzle": str,
        "solution": str
    },
    nrows=500000
)

In [30]:
print(type(df.loc[0, "puzzle"]))
print(len(df.loc[0, "puzzle"]))

print(type(df.loc[0, "solution"]))
print(len(df.loc[0, "solution"]))

<class 'str'>
81
<class 'str'>
81


In [31]:
#convertimos cadenas en arrays

def string_a_array(s):
    return np.array([int(char) for char in s], dtype=np.int32)

X = np.array([string_a_array(p) for p in df["puzzle"]])
y = np.array([string_a_array(s) for s in df["solution"]])

print(X.shape)
print(y.shape)
print(X[0])
print(y[0])

(500000, 81)
(500000, 81)
[0 7 0 0 0 0 0 4 3 0 4 0 0 0 9 6 1 0 8 0 0 6 3 4 9 0 0 0 9 4 0 5 2 0 0 0 3
 5 8 4 6 0 0 2 0 0 0 0 8 0 0 5 3 0 0 8 0 0 7 0 0 9 1 9 0 2 1 0 0 0 0 5 0 0
 7 0 4 0 8 0 2]
[6 7 9 5 1 8 2 4 3 5 4 3 7 2 9 6 1 8 8 2 1 6 3 4 9 5 7 7 9 4 3 5 2 1 8 6 3
 5 8 4 6 1 7 2 9 2 1 6 8 9 7 5 3 4 4 8 5 2 7 6 3 9 1 9 6 2 1 8 3 4 7 5 1 3
 7 9 4 5 8 6 2]


In [32]:
#preparacion/normalizacion datos

X_model = X.astype("float32") / 9.0
y_model = y.astype("int32")

print(X_model.shape)
print(y_model.shape)
print(X_model[0])
print(y_model[0])

(500000, 81)
(500000, 81)
[0.         0.7777778  0.         0.         0.         0.
 0.         0.44444445 0.33333334 0.         0.44444445 0.
 0.         0.         1.         0.6666667  0.11111111 0.
 0.8888889  0.         0.         0.6666667  0.33333334 0.44444445
 1.         0.         0.         0.         1.         0.44444445
 0.         0.5555556  0.22222222 0.         0.         0.
 0.33333334 0.5555556  0.8888889  0.44444445 0.6666667  0.
 0.         0.22222222 0.         0.         0.         0.
 0.8888889  0.         0.         0.5555556  0.33333334 0.
 0.         0.8888889  0.         0.         0.7777778  0.
 0.         1.         0.11111111 1.         0.         0.22222222
 0.11111111 0.         0.         0.         0.         0.5555556
 0.         0.         0.7777778  0.         0.44444445 0.
 0.8888889  0.         0.22222222]
[6 7 9 5 1 8 2 4 3 5 4 3 7 2 9 6 1 8 8 2 1 6 3 4 9 5 7 7 9 4 3 5 2 1 8 6 3
 5 8 4 6 1 7 2 9 2 1 6 8 9 7 5 3 4 4 8 5 2 7 6 3 9 1 9 6 2 1 8 3 4

In [33]:
#separacion train y test


X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(400000, 81) (100000, 81)
(400000, 81) (100000, 81)


In [34]:
modelo_solver = models.Sequential([
    layers.Input(shape=(81,)),

    layers.Dense(256, activation="relu"),
    layers.Dense(512, activation="relu"),
    layers.Dense(512, activation="relu"),
    layers.Dense(729),

    layers.Reshape((81, 9)),
    layers.Activation("softmax")
])

modelo_solver.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

modelo_solver.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 256)            │        20,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 729)            │       373,977 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_2 (Reshape)             │ (None, 81, 9)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 81, 9)          │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 789,209 (3.01 MB)

 Trainable params: 789,209 (3.01 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
#ajustar etiquetas

y_train_model = y_train - 1
y_test_model = y_test - 1

print(y_train_model.min(), y_train_model.max())
print(y_test_model.min(), y_test_model.max())


0 8
0 8


In [36]:
#entrenar 

history = modelo_solver.fit(
    X_train,
    y_train_model,
    epochs=25,
    batch_size=256,
    validation_data=(X_test, y_test_model)
)


Epoch 1/25
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - accuracy: 0.1751 - loss: 2.0710 - val_accuracy: 0.2134 - val_loss: 1.9491
Epoch 2/25
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - accuracy: 0.2574 - loss: 1.8655 - val_accuracy: 0.2934 - val_loss: 1.7976
Epoch 3/25
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - accuracy: 0.3249 - loss: 1.7325 - val_accuracy: 0.3490 - val_loss: 1.6771
Epoch 4/25
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - accuracy: 0.3682 - loss: 1.6340 - val_accuracy: 0.3802 - val_loss: 1.6072
Epoch 5/25
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - accuracy: 0.3897 - loss: 1.5835 - val_accuracy: 0.3953 - val_loss: 1.5679
Epoch 6/25
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - accuracy: 0.4015 - loss: 1.5552 - val_accuracy: 0.4037 - val_loss: 1.5521
Epoch 7/25
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - accuracy: 0.4092 - loss: 1.5349 - val_accuracy: 0.4122 - val_loss: 1.5273
Epoch 8/25
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - accuracy: 0.4149 -

In [ ]:
#primera prediccion: seleccionamos el primer sudoku del conjunto de test

sudoku_entrada = X_test[0]
sudoku_real = y_test[0]

# Predecimos
pred = modelo_solver.predict(
    sudoku_entrada.reshape(1, 81),
    verbose=0
)

# Nos quedamos con la clase más probable
pred = np.argmax(pred, axis=-1) + 1

pred = pred.reshape(9, 9)

# Recuperamos el sudoku original en formato 9x9
sudoku_entrada_matriz = (sudoku_entrada.reshape(9, 9) * 9).astype(int)

# Mantenemos los valores originales distintos de cero
pred_corregida = pred.copy()
pred_corregida[sudoku_entrada_matriz != 0] = sudoku_entrada_matriz[sudoku_entrada_matriz != 0]

print("Sudoku de entrada:")
print(sudoku_entrada_matriz)

print("\nSolución real:")
print(sudoku_real.reshape(9, 9))

print("\nPredicción original del modelo:")
print(pred)

print("\nPredicción corregida manteniendo las pistas originales:")
print(pred_corregida)

Sudoku de entrada:
[[9. 5. 0. 0. 1. 0. 0. 2. 0.]
 [0. 0. 6. 0. 5. 2. 0. 0. 3.]
 [0. 2. 0. 3. 4. 0. 0. 0. 9.]
 [6. 4. 2. 0. 0. 0. 0. 0. 0.]
 [3. 7. 0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 5. 0. 0. 0. 0. 6. 0.]
 [0. 3. 0. 2. 9. 0. 0. 0. 4.]
 [0. 0. 9. 4. 0. 0. 0. 0. 6.]
 [5. 0. 0. 0. 3. 1. 0. 8. 0.]]

Solución real:
[[9 5 3 6 1 7 4 2 8]
 [4 8 6 9 5 2 1 7 3]
 [7 2 1 3 4 8 6 5 9]
 [6 4 2 1 7 3 8 9 5]
 [3 7 8 5 6 9 2 4 1]
 [1 9 5 8 2 4 3 6 7]
 [8 3 7 2 9 6 5 1 4]
 [2 1 9 4 8 5 7 3 6]
 [5 6 4 7 3 1 9 8 2]]

Predicción del modelo:
[[9 5 2 7 7 8 6 4 6]
 [5 8 6 9 8 9 4 4 4]
 [1 4 1 3 5 6 6 6 9]
 [6 9 3 7 6 9 9 9 1]
 [3 7 9 5 5 5 8 9 1]
 [1 9 5 2 7 4 3 6 7]
 [6 1 7 8 9 7 6 1 4]
 [7 3 9 5 2 6 5 2 6]
 [5 9 6 4 3 2 9 8 6]]


In [ ]:
#guardamos modelo

modelo_solver.save("modelo_solver_sudoku_v2.keras")